# EfficientNet-B0 для предсказания PM2.5

Обучаем регрессионную модель на двух датасетах:
- **PM25Vision** — фото Mapillary + метки WAQI, 11 114 изображений
- **India/Nepal AQI Dataset** — уличные фото из Индии и Непала, 7 833 изображений

### Структура ноутбука
1. Установка зависимостей
2. Конфигурация путей
3. Загрузка и исследование данных
4. Dataset / DataLoader
5. Модель
6. Обучение (head-only → fine-tune)
7. Сохранение модели
8. Функция инференса

## 1. Установка зависимостей

In [4]:
!pip install -q torch torchvision datasets pillow scikit-learn

## 2. Импорты и конфигурация

In [5]:
import os
import copy
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms as T
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from PIL import Image

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# AFTER
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("GPUs available:", torch.cuda.device_count())

PyTorch version: 2.10.0+cu128
CUDA available: True
Device: cuda
GPUs available: 2


## 3. Пути к данным

In [22]:
# ── PM25Vision ────────────────────────────────────────────────────────────────
KAGGLE_INPUT    = "/kaggle/input"
PM25_ROOT       = os.path.join(KAGGLE_INPUT, "datasets", "deadcardassian", "pm25vision")
PM25_TRAIN_DIR  = os.path.join(PM25_ROOT, "train")
PM25_TEST_DIR   = os.path.join(PM25_ROOT, "test")
PM25_IMAGES_DIR = os.path.join(PM25_TRAIN_DIR, "images")
PM25_CSV        = os.path.join(PM25_TRAIN_DIR, "metadata.csv")
PM25_IMAGE_COL  = "filename"
PM25_LABEL_COL  = "pm25"

# ── India / Nepal dataset ──────────────────────────────────────────────────────
# CSVs are in Dataset_for_AQI_Classification
INDIA_CSV_ROOT   = "/kaggle/input/datasets/adarshrouniyar/air-pollution-image-dataset-from-india-and-nepal/Dataset_for_AQI_Classification/Dataset_for_AQI_Classification"
INDIA_TRAIN_CSV  = os.path.join(INDIA_CSV_ROOT, "train_data.csv")
INDIA_VAL_CSV    = os.path.join(INDIA_CSV_ROOT, "val_data.csv")

# Images are in Air Pollution Image Dataset/Combined_Dataset/All_img
INDIA_IMAGES_DIR = "/kaggle/input/datasets/adarshrouniyar/air-pollution-image-dataset-from-india-and-nepal/Air Pollution Image Dataset/Air Pollution Image Dataset/Combined_Dataset/All_img"

INDIA_IMAGE_COL  = "Filename"
INDIA_LABEL_COL  = "PM2.5"
print("PM25Vision train images:", PM25_IMAGES_DIR)
print("India/Nepal images:     ", INDIA_IMAGES_DIR)

PM25Vision train images: /kaggle/input/datasets/deadcardassian/pm25vision/train/images
India/Nepal images:      /kaggle/input/datasets/adarshrouniyar/air-pollution-image-dataset-from-india-and-nepal/Air Pollution Image Dataset/Air Pollution Image Dataset/Combined_Dataset/All_img


In [23]:
print("Train CSV exists:", os.path.exists(INDIA_TRAIN_CSV))
print("Val CSV exists:  ", os.path.exists(INDIA_VAL_CSV))

india_train_df = pd.read_csv(INDIA_TRAIN_CSV)
sample = os.path.join(INDIA_IMAGES_DIR, india_train_df.iloc[0]["Filename"])
print("Sample image exists:", os.path.exists(sample))


Train CSV exists: True
Val CSV exists:   True
Sample image exists: True


## 4. Загрузка и исследование данных

In [24]:
# ── PM25Vision ────────────────────────────────────────────────────────────────
pm25_df = pd.read_csv(PM25_CSV)
print(f"PM25Vision shape: {pm25_df.shape}")
print(pm25_df[PM25_LABEL_COL].describe())
pm25_df.head()

PM25Vision shape: (8298, 12)
count    8298.000000
mean      122.959749
std        88.537223
min         1.000000
25%        54.000000
50%       112.000000
75%       165.000000
max       530.000000
Name: pm25, dtype: float64


,image_id,station_id,captured_at,camera_angle,longitude,latitude,quality_score,downloaded_at,pm25,filename,quality,pm25_bin
0,1299925057505266,13051,2024-01-17,NaN,121.293349,25.027580,NaN,2025-09-12T08:00:56.871094,39.0,1299925057505266.jpg,good,0–50
1,924249681642820,3705,2017-12-30,NaN,139.235777,35.354056,NaN,2025-09-07T22:39:44.846055,19.0,924249681642820.jpg,good,0–50
2,498043544721455,3278,2020-01-02,NaN,145.714003,-40.989167,NaN,2025-09-07T18:25:10.505759,9.0,498043544721455.jpg,good,0–50
3,944035996420125,11580,2019-06-27,NaN,32.024932,-28.747296,NaN,2025-09-11T21:11:04.811658,31.0,944035996420125.jpg,good,0–50
4,886419591948919,6894,2020-04-24,NaN,4.833378,43.933108,NaN,2025-09-09T14:24:31.876920,41.0,886419591948919.jpg,good,0–50


/kaggle/input/datasets/adarshrouniyar/air-pollution-image-dataset-from-india-and-nepal/Dataset_for_AQI_Classification/Dataset_for_AQI_Classification/val_data.csv
/kaggle/input/datasets/adarshrouniyar/air-pollution-image-dataset-from-india-and-nepal/Dataset_for_AQI_Classification/Dataset_for_AQI_Classification/testing_data.csv
/kaggle/input/datasets/adarshrouniyar/air-pollution-image-dataset-from-india-and-nepal/Dataset_for_AQI_Classification/Dataset_for_AQI_Classification/train_data.csv
/kaggle/input/datasets/adarshrouniyar/air-pollution-image-dataset-from-india-and-nepal/Dataset_for_AQI_Classification/Dataset_for_AQI_Classification/Sample result file.csv
/kaggle/input/datasets/adarshrouniyar/air-pollution-image-dataset-from-india-and-nepal/Air Pollution Image Dataset/Air Pollution Image Dataset/Country_wise_Dataset/Nepal/Biratnagar/Biratnagar_AQI_All_info.csv
/kaggle/input/datasets/adarshrouniyar/air-pollution-image-dataset-from-india-and-nepal/Air Pollution Image Dataset/Air Pollutio

KeyboardInterrupt: 

In [26]:
# ── India / Nepal ──────────────────────────────────────────────────────────────
india_train_df = pd.read_csv(INDIA_TRAIN_CSV)
india_val_df   = pd.read_csv(INDIA_VAL_CSV)

print(f"India train shape: {india_train_df.shape}")
print(f"India val shape:   {india_val_df.shape}")
print(india_train_df[INDIA_LABEL_COL].describe())
india_train_df.head()

India train shape: (7833, 14)
India val shape:   (1959, 14)
count    7833.000000
mean      142.615333
std       130.099442
min         4.000000
25%        35.000000
50%        70.080000
75%       257.000000
max       500.000000
Name: PM2.5, dtype: float64


,Location,Filename,Year,Month,Day,Hour,AQI,PM2.5,PM10,O3,CO,SO2,NO2,AQI_Class
0,Tamil Nadu,TN_UnFSG_2023-03-02-08.30-1.jpg,2023,3,2,8:30,119,69.00,106.00,8.00,358.00,22.0,26.00,c_Unhealthy_for_Sensitive_Groups
1,Bengaluru,BENGR_Mod_2023-02-24-08.30-1-166.jpg,2023,2,24,8:30,68,32.00,61.00,26.00,228.00,5.0,21.00,b_Moderate
2,"Biratnagar, Nepal",BIR_UNFSG_VF_2023-02-03-15.00-2-24.jpg,2023,2,3,15:00,141,47.96,68.92,65.57,0.41,2.8,2.51,c_Unhealthy_for_Sensitive_Groups
3,Mumbai,MH_UnFSG_2023-03-10-16.00-1-76.jpg,2023,3,10,16:00,141,72.00,108.00,NaN,NaN,NaN,63.00,c_Unhealthy_for_Sensitive_Groups
4,"ITO, Delhi",DEL_SEV_2023-02-07-14.00-2-8.jpg,2023,2,7,14:00,449,337.00,198.00,23.00,24.00,13.0,57.00,f_Severe


In [27]:
# Сравнение распределений PM2.5 между датасетами
print("=== PM25Vision (full) ===")
print(pm25_df[PM25_LABEL_COL].describe())

print("\n=== India/Nepal (train) ===")
print(india_train_df[INDIA_LABEL_COL].describe())

# Проверка пути к изображениям (оба датасета)
pm25_sample  = os.path.join(PM25_IMAGES_DIR, pm25_df.iloc[0][PM25_IMAGE_COL])
india_sample = os.path.join(INDIA_IMAGES_DIR, india_train_df.iloc[0][INDIA_IMAGE_COL])
print(f"\nPM25Vision sample exists:   {os.path.exists(pm25_sample)}  → {pm25_sample}")
print(f"India/Nepal sample exists:  {os.path.exists(india_sample)} → {india_sample}")

=== PM25Vision (full) ===
count    8298.000000
mean      122.959749
std        88.537223
min         1.000000
25%        54.000000
50%       112.000000
75%       165.000000
max       530.000000
Name: pm25, dtype: float64

=== India/Nepal (train) ===
count    7833.000000
mean      142.615333
std       130.099442
min         4.000000
25%        35.000000
50%        70.080000
75%       257.000000
max       500.000000
Name: PM2.5, dtype: float64

PM25Vision sample exists:   True  → /kaggle/input/datasets/deadcardassian/pm25vision/train/images/1299925057505266.jpg
India/Nepal sample exists:  True → /kaggle/input/datasets/adarshrouniyar/air-pollution-image-dataset-from-india-and-nepal/Air Pollution Image Dataset/Air Pollution Image Dataset/Combined_Dataset/All_img/TN_UnFSG_2023-03-02-08.30-1.jpg


## 5. Train / Val split для PM25Vision

In [28]:
pm25_train_df, pm25_val_df = train_test_split(
    pm25_df, test_size=0.15, random_state=42
)
print(f"PM25Vision → Train: {len(pm25_train_df)} | Val: {len(pm25_val_df)}")

PM25Vision → Train: 7053 | Val: 1245


## 6. Transforms и Dataset

In [29]:
IMG_SIZE = 224

train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class GenericPM25Dataset(Dataset):
    """
    Универсальный Dataset для любого датафрейма с колонками
    image_col (имя файла) и label_col (значение PM2.5).

    Args:
        dataframe  : pd.DataFrame
        images_dir : str  — папка, в которой лежат изображения
        transform  : torchvision transform
        image_col  : str  — имя колонки с именем файла
        label_col  : str  — имя колонки с меткой PM2.5
    """

    def __init__(self, dataframe, images_dir, transform, image_col, label_col):
        self.df         = dataframe.reset_index(drop=True)
        self.images_dir = images_dir
        self.transform  = transform
        self.image_col  = image_col
        self.label_col  = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.images_dir, row[self.image_col])
        img      = Image.open(img_path).convert("RGB")
        img      = self.transform(img)
        label    = torch.tensor(float(row[self.label_col]), dtype=torch.float32)
        return img, label


print("GenericPM25Dataset defined ✓")

GenericPM25Dataset defined ✓


In [30]:
BATCH_SIZE = 64

# ── PM25Vision datasets ───────────────────────────────────────────────────────
pm25_train_ds = GenericPM25Dataset(
    pm25_train_df, PM25_IMAGES_DIR, train_transform, PM25_IMAGE_COL, PM25_LABEL_COL
)
pm25_val_ds = GenericPM25Dataset(
    pm25_val_df, PM25_IMAGES_DIR, eval_transform, PM25_IMAGE_COL, PM25_LABEL_COL
)

# ── India/Nepal datasets ──────────────────────────────────────────────────────
# Удаляем строки с NaN в колонке PM2.5
india_train_clean = india_train_df.dropna(subset=[INDIA_LABEL_COL]).reset_index(drop=True)
india_val_clean   = india_val_df.dropna(subset=[INDIA_LABEL_COL]).reset_index(drop=True)

print(f"India train after dropna: {len(india_train_clean)} (было {len(india_train_df)})")
print(f"India val after dropna:   {len(india_val_clean)} (было {len(india_val_df)})")

india_train_ds = GenericPM25Dataset(
    india_train_clean, INDIA_IMAGES_DIR, train_transform, INDIA_IMAGE_COL, INDIA_LABEL_COL
)
india_val_ds = GenericPM25Dataset(
    india_val_clean, INDIA_IMAGES_DIR, eval_transform, INDIA_IMAGE_COL, INDIA_LABEL_COL
)

# ── Объединённые датасеты ─────────────────────────────────────────────────────
combined_train_ds = ConcatDataset([pm25_train_ds, india_train_ds])
combined_val_ds   = ConcatDataset([pm25_val_ds,   india_val_ds])

print(f"\nCombined train: {len(combined_train_ds)} | Combined val: {len(combined_val_ds)}")

combined_train_loader = DataLoader(
    combined_train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True
)
combined_val_loader = DataLoader(
    combined_val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)

print("DataLoaders ready ✓")

India train after dropna: 7833 (было 7833)
India val after dropna:   1959 (было 1959)

Combined train: 14886 | Combined val: 3204
DataLoaders ready ✓


## 7. Модель

In [31]:
class PM25Estimator(nn.Module):
    """
    EfficientNet-B0 backbone + регрессионная голова для предсказания PM2.5.

    Args:
        freeze_backbone : bool — заморозить ли веса backbone при инициализации
    """

    def __init__(self, freeze_backbone: bool = True):
        super().__init__()
        backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        backbone.classifier = nn.Identity()   # убираем оригинальный классификатор
        self.backbone = backbone

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        # Регрессионная голова
        self.head = nn.Sequential(
            nn.Linear(1280, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)              # (B, 1280)
        return self.head(features).squeeze(1)    # (B,)

    def unfreeze_backbone(self):
        """Разморозить backbone для fine-tuning."""
        for p in self.backbone.parameters():
            p.requires_grad = True


model = PM25Estimator(freeze_backbone=True)

# ── Multi-GPU via DataParallel ────────────────────────────────────────────────
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)   # wraps model for parallel execution

model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable:,}  (head only)")

Using 2 GPUs!
Total params:     4,351,997
Trainable params: 344,449  (head only)


## 8. Функции обучения и оценки

In [32]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        preds = model(imgs)
        loss  = criterion(preds, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs  = imgs.to(device)
        preds = model(imgs).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())
    mae = mean_absolute_error(all_labels, all_preds)
    r2  = r2_score(all_labels, all_preds)
    return mae, r2


print("train_one_epoch / evaluate defined ✓")

train_one_epoch / evaluate defined ✓


In [33]:
import os

# Walk the India/Nepal dataset to find where images actually are
print("Searching for image folders in India/Nepal dataset...")
for root, dirs, files in os.walk(INDIA_ROOT):
    jpgs = [f for f in files if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    if jpgs:
        print(f"  Found {len(jpgs)} images in: {root}")
        print(f"  Sample: {jpgs[0]}")
        break  # show first folder found

Searching for image folders in India/Nepal dataset...
  Found 1 images in: /kaggle/input/datasets/adarshrouniyar/air-pollution-image-dataset-from-india-and-nepal/Air Pollution Image Dataset/Air Pollution Image Dataset
  Sample: Data Distribution.png


## 9. Фаза 1 — обучение только «головы» (backbone заморожен)

In [34]:
EPOCHS_PHASE1 = 15
LR_PHASE1     = 1e-3

criterion  = nn.MSELoss()
head_params = model.module.head.parameters() if hasattr(model, "module") else model.head.parameters()
optimizer1 = torch.optim.Adam(head_params, lr=LR_PHASE1)

best_mae   = float("inf")
best_state = None
history    = []

print(f"{'Epoch':>5} | {'train_loss':>10} | {'val_MAE':>8} | {'val_R2':>7}")
print("-" * 42)

for epoch in range(1, EPOCHS_PHASE1 + 1):
    train_loss      = train_one_epoch(model, combined_train_loader, optimizer1, criterion, device)
    val_mae, val_r2 = evaluate(model, combined_val_loader, device)
    history.append((epoch, train_loss, val_mae, val_r2))

    marker = ""
    if val_mae < best_mae:
        best_mae   = val_mae
        best_state = copy.deepcopy(model.state_dict())
        marker     = "  ← best"

    print(f"{epoch:>5} | {train_loss:>10.3f} | {val_mae:>8.2f} | {val_r2:>7.3f}{marker}")

print(f"\nФаза 1 завершена. Best val_MAE = {best_mae:.2f}")

Epoch | train_loss |  val_MAE |  val_R2
------------------------------------------
    1 |  13555.240 |    67.79 |   0.403  ← best
    2 |   7170.525 |    53.24 |   0.614  ← best
    3 |   5752.090 |    47.95 |   0.670  ← best
    4 |   5147.227 |    46.69 |   0.688  ← best
    5 |   4762.102 |    45.49 |   0.707  ← best
    6 |   4480.004 |    43.62 |   0.726  ← best
    7 |   4323.197 |    42.33 |   0.737  ← best
    8 |   4179.172 |    42.60 |   0.741
    9 |   3968.534 |    41.35 |   0.752  ← best
   10 |   3837.430 |    41.37 |   0.749
   11 |   3733.766 |    40.56 |   0.752  ← best
   12 |   3620.575 |    39.47 |   0.767  ← best
   13 |   3547.660 |    39.53 |   0.766
   14 |   3482.658 |    39.25 |   0.767  ← best
   15 |   3345.400 |    40.64 |   0.759

Фаза 1 завершена. Best val_MAE = 39.25


## 10. Фаза 2 — Fine-tuning всей сети

Загружаем лучший checkpoint с фазы 1 и дообучаем всю сеть на меньшем LR.

In [35]:
# Восстанавливаем лучшие веса с фазы 1
model.load_state_dict(best_state) 
# Размораживаем backbone
if hasattr(model, "module"):
    model.module.unfreeze_backbone()
else:
    model.unfreeze_backbone()


total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params after unfreeze: {total_trainable:,}")

FT_EPOCHS  = 5
LR_FT      = 1e-5
optimizer2 = torch.optim.Adam(model.parameters(), lr=LR_FT)

print(f"\n{'Epoch':>5} | {'train_loss':>10} | {'val_MAE':>8} | {'val_R2':>7}")
print("-" * 42)

for epoch in range(1, FT_EPOCHS + 1):
    train_loss      = train_one_epoch(model, combined_train_loader, optimizer2, criterion, device)
    val_mae, val_r2 = evaluate(model, combined_val_loader, device)
    history.append((EPOCHS_PHASE1 + epoch, train_loss, val_mae, val_r2))

    marker = ""
    if val_mae < best_mae:
        best_mae   = val_mae
        best_state = copy.deepcopy(model.state_dict())
        marker     = "  ← best"

    print(f"{epoch:>5} | {train_loss:>10.3f} | {val_mae:>8.2f} | {val_r2:>7.3f}{marker}")

print(f"\nFine-tuning завершён. Best overall val_MAE = {best_mae:.2f}")

Trainable params after unfreeze: 4,351,997

Epoch | train_loss |  val_MAE |  val_R2
------------------------------------------
    1 |   3178.160 |    37.56 |   0.787  ← best
    2 |   2903.129 |    36.90 |   0.791  ← best
    3 |   2619.728 |    36.83 |   0.791  ← best
    4 |   2491.231 |    35.81 |   0.802  ← best
    5 |   2447.519 |    35.28 |   0.808  ← best

Fine-tuning завершён. Best overall val_MAE = 35.28


## 11. Сохранение лучшей модели

In [36]:
OUTPUT_PATH = "/kaggle/working/pm25_efficientnet_b0.pt"

model.load_state_dict(best_state)
torch.save(model.state_dict(), OUTPUT_PATH)
print(f"Модель сохранена: {OUTPUT_PATH}")
print(f"Размер файла:     {os.path.getsize(OUTPUT_PATH) / 1e6:.1f} MB")

Модель сохранена: /kaggle/working/pm25_efficientnet_b0.pt
Размер файла:     17.7 MB


## 12. Лог обучения

In [37]:
log_df = pd.DataFrame(history, columns=["epoch", "train_loss", "val_MAE", "val_R2"])
print(log_df.to_string(index=False))

 epoch   train_loss   val_MAE   val_R2
     1 13555.239614 67.792865 0.403455
     2  7170.524739 53.236005 0.613518
     3  5752.090322 47.953621 0.669537
     4  5147.227259 46.690715 0.688489
     5  4762.102328 45.493189 0.707397
     6  4480.003721 43.615040 0.726264
     7  4323.197082 42.327538 0.736708
     8  4179.172079 42.604090 0.741228
     9  3968.533668 41.348101 0.751817
    10  3837.430239 41.368541 0.749357
    11  3733.765584 40.558165 0.752249
    12  3620.574506 39.472230 0.766752
    13  3547.659557 39.531359 0.766278
    14  3482.657653 39.247493 0.767425
    15  3345.400475 40.640692 0.759151
    16  3178.159846 37.564046 0.787411
    17  2903.128565 36.901868 0.791375
    18  2619.728009 36.827767 0.790778
    19  2491.231413 35.807722 0.801991
    20  2447.519161 35.275158 0.808446


## 13. Инференс на одном изображении

In [38]:
@torch.no_grad()
def predict_pm25(image_path: str, model: nn.Module, device: torch.device) -> float:
    """
    Предсказывает PM2.5 по пути к изображению.

    Args:
        image_path : путь к изображению (jpg / png / ...)
        model      : обученная PM25Estimator
        device     : torch.device

    Returns:
        Предсказанное значение PM2.5 (float)
    """
    model.eval()
    img = Image.open(image_path).convert("RGB")
    x   = eval_transform(img).unsqueeze(0).to(device)
    return model(x).item()


# ── Пример использования ───────────────────────────────────────────────────────
sample_row  = pm25_val_df.sample(1).iloc[0]
sample_path = os.path.join(PM25_IMAGES_DIR, sample_row[PM25_IMAGE_COL])
true_value  = sample_row[PM25_LABEL_COL]

pred_value = predict_pm25(sample_path, model, device)
print(f"Image      : {sample_path}")
print(f"True PM2.5 : {true_value:.1f} µg/m³")
print(f"Predicted  : {pred_value:.1f} µg/m³")
print(f"Error      : {abs(true_value - pred_value):.1f} µg/m³")

Image      : /kaggle/input/datasets/deadcardassian/pm25vision/train/images/453766226324799.jpg
True PM2.5 : 224.0 µg/m³
Predicted  : 242.6 µg/m³
Error      : 18.6 µg/m³
